# RiskLens: LightGBM model

**Goal of this notebook:** train a gradient boosting model and see how much better it is than the logistic regression baseline (ROC AUC 0.744). This time I also use the text columns that the baseline skipped.

**Data:** `train_merged.csv` (307,511 applicants, 134 columns), same validation rows as notebook 05

**What I'll do**
1. Load the data and prepare the text columns
2. Use the same split as the baseline
3. Train with early stopping
4. Evaluate and compare with the baseline
5. Look at feature importance
6. Save the model

## 1. Data and features

In [ ]:
# Step 1: Setup and data 
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, precision_score, confusion_matrix
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

df = pd.read_csv("../data/processed/train_merged.csv")
print(df.shape)

(307511, 134)


In [ ]:
y = df["TARGET"]
X = df.drop(columns=["TARGET", "SK_ID_CURR"])
X = X.replace([np.inf, -np.inf], np.nan)

# text columns -> category type, lightgbm can use these directly
cat_cols = X.select_dtypes(exclude="number").columns
for c in cat_cols:
    X[c] = X[c].astype("category")

print(X.shape, len(cat_cols))

C:\Users\HP\AppData\Local\Temp\ipykernel_5124\669988290.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns


(307511, 132) 16


The table has **132 feature columns**, and **16 of them are text** (contract type, gender, income type, occupation, organization type, ...). I converted them to the `category` type, because LightGBM can use them directly. That means no one-hot encoding is needed. It also doesn't need an imputer or a scaler, since it handles missing values and column scales by itself.

## 2. Same split as the baseline

In [3]:
# Step 2: Same split as the baseline
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(X_train.shape, X_val.shape)
print(y_val.mean())

(246008, 132) (61503, 132)
0.08072776937710356


Same split as notebook 05 (`random_state=42`): **246,008 train rows and 61,503 validation rows**, with 8.07% defaults in the validation set. Using the same rows means the comparison with the baseline is fair.

I also took **10% of the train set (24,601 rows)** out just for early stopping, so the model trains on 221,407 rows. The validation set stays untouched until the final evaluation. If the model had chosen its number of trees by looking at the validation set, the final score would be a bit too optimistic.

## 3. Training

I used mostly default settings (learning rate 0.05, 31 leaves, balanced class weights like in the baseline) and up to 2,000 trees. **Early stopping** stops training when the AUC on the early-stopping set hasn't improved for 100 rounds. I haven't tuned any hyperparameters.

In [4]:
# a small part of train is kept just for early stopping,
# so the validation set stays untouched until the final evaluation
X_tr, X_es, y_tr, y_es = train_test_split(
    X_train, y_train, test_size=0.1, stratify=y_train, random_state=42
)

In [5]:
# Step 3: Train
model = LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=31,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_es, y_es)],
    eval_metric="auc",
    callbacks=[early_stopping(100), log_evaluation(100)]
)

print("best number of trees:", model.best_iteration_)

c:\Users\HP\Desktop\Projects\risklens-explainable-credit-risk\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 17874, number of negative: 203533
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.099272 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 13457
[LightGBM] [Info] Number of data points in the train set: 221407, number of used features: 128
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.755729	valid_0's binary_logloss: 0.574911
[200]	valid_0's auc: 0.758166	valid_0's binary_logloss: 0.557729
Early stopping, best iteration is:
[167]	valid_0's auc: 0.758316	valid_0's binary_logloss: 0.562343
best number of trees: 167


The model stopped early. The best number of trees was **167** (AUC of 0.758 on the early-stopping set), and it stopped 100 rounds later because the AUC wasn't improving anymore. So 2,000 trees would have been far too many and would have just overfit.

LightGBM says it used **128 of the 132 columns**. I haven't checked which 4 weren't used. The two warnings in the output (pandas `select_dtypes` and LightGBM `eval_set`) are about future versions and don't change the results.

## 4. Results on the validation set

Same metrics and same 0.5 threshold as the baseline, so the two can be compared directly.

In [6]:
# Step 4: Evaluate (same metrics as the baseline)
probs = model.predict_proba(X_val)[:, 1]

print("ROC AUC:", roc_auc_score(y_val, probs))
print("PR AUC:", average_precision_score(y_val, probs))

preds = (probs > 0.5).astype(int)
print("recall:", recall_score(y_val, preds))
print("precision:", precision_score(y_val, preds))
print(confusion_matrix(y_val, preds))

ROC AUC: 0.7634728393601152
PR AUC: 0.25806049952491167
recall: 0.6737160120845922
precision: 0.1737752610525222
[[40634 15904]
 [ 1620  3345]]


| Metric | Logistic regression | LightGBM |
|---|---|---|
| ROC AUC | 0.744 | **0.763** |
| PR AUC | 0.224 | **0.258** |
| Recall (default class) | 0.670 | 0.674 |
| Precision (default class) | 0.159 | 0.174 |
| Defaulters caught | 3,325 | 3,345 |
| People flagged | 20,868 | 19,249 |
| False alarms | 17,543 | 15,904 |

| | Predicted: no default | Predicted: default |
|---|---|---|
| **Actually no default** | 40,634 | 15,904 |
| **Actually default** | 1,620 | 3,345 |

- **The improvement is real but modest.** ROC AUC went from 0.744 to 0.763 (+0.019), and PR AUC from 0.224 to 0.258, around 15% higher. PR AUC is now about **3.2 times** what a random model would get (0.081).
- **Recall is almost the same** (67.4% vs 67.0%, 20 more defaulters caught), but there are about **1,600 fewer false alarms** (roughly 9% fewer). So for the same recall, the model is a bit more precise.
- It still flags **31% of all applicants** and only 17% of those flagged really default, so I wouldn't use this to make loan decisions as it is.
- The AUC on the validation set (0.763) is close to the early-stopping AUC (0.758), so nothing suggests the model is doing much better on one part of the data than another.
- I didn't tune anything. I think the more important next steps are explaining the model and checking fairness, not squeezing out a few more AUC points.

## 5. Which features does the model use?

I used importance by **gain**, which measures how much each feature improves the model's splits in total. It tells me how much a feature is used, but not whether it raises or lowers the risk. SHAP will show that.

In [7]:
#Step 5: Feature importance
imp = pd.Series(
    model.booster_.feature_importance(importance_type="gain"),
    index=X.columns
).sort_values(ascending=False)

print(imp.head(15))

EXT_SOURCE_3           182402.486420
EXT_SOURCE_2           141250.321926
ORGANIZATION_TYPE       77094.845118
EXT_SOURCE_1            50196.148817
DAYS_EMPLOYED           21677.863119
AMT_CREDIT              17793.261574
OCCUPATION_TYPE         15437.782642
prev_refused_rate       15398.276045
AMT_GOODS_PRICE         13262.328070
OWN_CAR_AGE             11939.192574
NAME_EDUCATION_TYPE     11728.880318
AMT_ANNUITY             11115.642710
CODE_GENDER             10973.251244
bureau_debt_sum         10331.674768
bureau_credit_sum        8668.871172
dtype: float64


| Rank | Feature | Gain | Type |
|---|---|---|---|
| 1 | EXT_SOURCE_3 | 182,402 | numeric |
| 2 | EXT_SOURCE_2 | 141,250 | numeric |
| 3 | ORGANIZATION_TYPE | 77,095 | text |
| 4 | EXT_SOURCE_1 | 50,196 | numeric |
| 5 | DAYS_EMPLOYED | 21,678 | numeric |
| 6 | AMT_CREDIT | 17,793 | numeric |
| 7 | OCCUPATION_TYPE | 15,438 | text |
| 8 | prev_refused_rate | 15,398 | my feature |
| 9 | AMT_GOODS_PRICE | 13,262 | numeric |
| 10 | OWN_CAR_AGE | 11,939 | numeric |
| 11 | NAME_EDUCATION_TYPE | 11,729 | text |
| 12 | AMT_ANNUITY | 11,116 | numeric |
| 13 | CODE_GENDER | 10,973 | text |
| 14 | bureau_debt_sum | 10,332 | my feature |
| 15 | bureau_credit_sum | 8,669 | my feature |

- **The three `EXT_SOURCE` columns are on top**, as in the baseline and in the correlations from notebook 01. They are clearly the most important by a wide margin.
- **Text columns matter.** `ORGANIZATION_TYPE` is third, and `OCCUPATION_TYPE`, `NAME_EDUCATION_TYPE` and `CODE_GENDER` are also in the top 15. The baseline couldn't use any of them, which probably explains part of the improvement. Gain can be inflated for text columns with many categories, so SHAP will give a more reliable picture.
- **Three of the features I built are in the top 15:** `prev_refused_rate` (8th), `bureau_debt_sum` (14th) and `bureau_credit_sum` (15th). So the aggregation work in notebooks 02 and 03 paid off. The ratio features and `AGE` are not in the top 15.
- **`CODE_GENDER` is in the top 15.** Gender is a sensitive attribute in credit decisions, so I'll look at it carefully in the fairness notebook, including what happens to the results without it.
- `AMT_CREDIT` and `AMT_GOODS_PRICE` are both high up. They're almost the same number for most loans, as I found in the baseline.

## 6. Saving the model

Saved with `joblib` to `models/lgbm_model.joblib` so I can load it for SHAP and later for the API. The `models/` folder is in `.gitignore`, so the file doesn't go to GitHub.

In [9]:
#Step 6: Save the model for the API later
import joblib
joblib.dump(model, "../models/lgbm_model.joblib")

['../models/lgbm_model.joblib']

## Summary

**What I did**
- Trained LightGBM on 132 features (116 numeric + 16 text columns), with early stopping (best: 167 trees)
- Evaluated on the same validation set as the baseline (61,503 rows)

**Results**
- ROC AUC **0.763** (baseline 0.744), PR AUC **0.258** (baseline 0.224)
- Recall 0.674, precision 0.174 at threshold 0.5
- Same recall as the baseline with about 1,600 fewer false alarms

**What I learned**
- The gain over logistic regression is real but modest, and I haven't tuned anything
- `EXT_SOURCE_1/2/3` dominate, text columns help, and 3 of my own features are in the top 15
- `CODE_GENDER` is among the most used features, so I need to check fairness

**Next:** SHAP to explain the model (`07_shap`), then fairness checks (`08_fairness`)